In [1]:
import string
import tkinter as tk
from tkinter import scrolledtext
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# 1. KNOWLEDGE BASE: FAQ COLLECTION

FAQ_DATA = [
    {
        "question": "What is your return policy?",
        "answer": "You can return any product within 30 days of purchase for a full refund."
    },
    {
        "question": "How long does shipping take?",
        "answer": "Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days."
    },
    {
        "question": "Do you ship internationally?",
        "answer": "Yes, we ship to over 50 countries worldwide. Shipping fees vary by location."
    },
    {
        "question": "How can I track my order?",
        "answer": "Once your order ships, we will email you a tracking number and a link to track it."
    },
    {
        "question": "What payment methods do you accept?",
        "answer": "We accept all major credit cards, PayPal, Apple Pay, and Google Pay."
    }
]

FAQ_QUESTIONS = [item["question"] for item in FAQ_DATA]
FAQ_ANSWERS = [item["answer"] for item in FAQ_DATA]






In [3]:
# 2. NLP TEXT PREPROCESSING

def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    return " ".join(tokens)

processed_faq_questions = [preprocess_text(q) for q in FAQ_QUESTIONS]

In [4]:
# 3. INTENT & SEMANTIC MATCHING ENGINE

def get_best_response(user_query, confidence_threshold=0.2):
    clean_query = preprocess_text(user_query)
    
    if not clean_query.strip():
        return "I didn't quite catch that. Could you please rephrase?"

    vectorizer = TfidfVectorizer()
    all_texts = processed_faq_questions + [clean_query]
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    faq_vectors = tfidf_matrix[:-1]
    query_vector = tfidf_matrix[-1]
    similarity_scores = cosine_similarity(query_vector, faq_vectors).flatten()
    
    best_match_idx = similarity_scores.argmax()
    highest_score = similarity_scores[best_match_idx]
    
    if highest_score >= confidence_threshold:
        return FAQ_ANSWERS[best_match_idx]
    else:
        return "I'm sorry, I couldn't find an answer to that question. Would you like to speak to a human agent?"




In [5]:
# 4. CHAT UI (DARK MODE TKINTER APP)

class ChatBotUI:
    def __init__(self, root):
        self.root = root
        self.root.title("FAQ Support Assistant")
        self.root.geometry("450x550")
        
        # --- DARK MODE PALETTE ---
        self.bg_main = "#1e1e24"       # Deep dark gray/black background
        self.bg_box = "#2a2a32"        # Lighter charcoal for text inputs/logs
        self.fg_text = "#e2e8f0"       # Off-white for crisp, readable text
        self.accent_blue = "#3b82f6"   # Electric blue button color
        self.accent_hover = "#2563eb"  # Darker blue on click/hover

        self.root.configure(bg=self.bg_main)

        # Layout Configuration
        self.root.grid_rowconfigure(0, weight=1)
        self.root.grid_columnconfigure(0, weight=1)

        # Dark Chat Log Box
        self.chat_log = scrolledtext.ScrolledText(
            root, state='disabled', wrap=tk.WORD, 
            bg=self.bg_box, fg=self.fg_text, 
            font=("Arial", 11), insertbackground=self.fg_text, bd=0, highlightthickness=0
        )
        self.chat_log.grid(row=0, column=0, columnspan=2, padx=15, pady=15, sticky="nsew")

        # Dark User Entry Field
        self.entry_field = tk.Entry(
            root, font=("Arial", 12), 
            bg=self.bg_box, fg=self.fg_text, 
            insertbackground=self.fg_text, bd=0, highlightthickness=0
        )
        self.entry_field.grid(row=1, column=0, padx=(15, 5), pady=(0, 15), ipady=10, sticky="ew")
        self.entry_field.bind("<Return>", self.send_message) 

        # High-Contrast Send Button
        self.send_button = tk.Button(
            root, text="Send", font=("Arial", 11, "bold"), 
            bg=self.accent_blue, fg="white",
            activebackground=self.accent_hover, activeforeground="white", 
            bd=0, highlightthickness=0, command=self.send_message
        )
        self.send_button.grid(row=1, column=1, padx=(5, 15), pady=(0, 15), ipady=8, ipadx=15, sticky="nsew")

        # Welcome Message
        self.display_message("Bot", "Hello! How can I help you today? Ask me about shipping, tracking, or returns.")

    def display_message(self, sender, text):
        self.chat_log.config(state='normal')
        self.chat_log.insert(tk.END, f"{sender}: {text}\n\n")
        self.chat_log.config(state='disabled')
        self.chat_log.yview(tk.END) 

    def send_message(self, event=None):
        user_text = self.entry_field.get().strip()
        
        if user_text:
            self.display_message("You", user_text)
            self.entry_field.delete(0, tk.END)
            
            bot_response = get_best_response(user_text)
            self.root.after(300, lambda: self.display_message("Bot", bot_response))




In [ ]:
# RUN APPLICATION

if __name__ == "__main__":
    window = tk.Tk()
    app = ChatBotUI(window)
    window.mainloop()